# 01 · Prepare data (run once)

Downloads FFHQ-256 (70k aligned faces, Kaggle) and COCO val2017 (5k scenes),
normalises everything to 256 px, detects 5-point face landmarks, fixes the
held-out eval split (300 FFHQ + 100 COCO, seed 42), and stores the result as
one tar on Drive. Later notebooks only unpack that tar.

Needs `kaggle.json` (kaggle.com → Settings → API → Create New Token), uploaded
when the cell asks. About 30–45 minutes on a T4 runtime.

In [ ]:
# ---- edit these ----
REPO_URL   = "https://github.com/AhnafAbdullah/Soteria-ImageCloaker.git"   # "" = copy from Drive
BRANCH     = "main"
DRIVE_ROOT = "/content/drive/MyDrive/soteria"   # runs, data tar and (optionally) code live here
# Private repo: add a Colab secret GITHUB_TOKEN. FLUX VAE: add HF_TOKEN.

In [ ]:
import os, shutil, subprocess
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive")
Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
CODE = "/content/Project-Soteria"

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

if REPO_URL:
    url = REPO_URL
    if secret("GITHUB_TOKEN"):
        url = url.replace("https://", f"https://{secret('GITHUB_TOKEN')}@")
    if Path(CODE).exists():
        subprocess.run(["git", "-C", CODE, "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "-b", BRANCH, url, CODE], check=True)
else:
    shutil.copytree(f"{DRIVE_ROOT}/code/Project-Soteria", CODE, dirs_exist_ok=True)

if secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = secret("HF_TOKEN")
os.chdir(CODE)
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install -r requirements-data.txt
from google.colab import files
if not Path("/root/.kaggle/kaggle.json").exists():
    up = files.upload()                      # choose kaggle.json
    Path("/root/.kaggle").mkdir(exist_ok=True)
    Path("/root/.kaggle/kaggle.json").write_bytes(up["kaggle.json"])
    os.chmod("/root/.kaggle/kaggle.json", 0o600)

In [ ]:
!mkdir -p /content/raw/ffhq /content/raw/coco
!kaggle datasets download -d xhlulu/flickrfaceshq-dataset-nvidia-resized-256px -p /content/raw/ffhq --unzip -q
!wget -q http://images.cocodataset.org/zips/val2017.zip -O /content/raw/coco/val2017.zip
!unzip -q /content/raw/coco/val2017.zip -d /content/raw/coco && rm /content/raw/coco/val2017.zip
!find /content/raw/ffhq -type f | wc -l && find /content/raw/coco -type f | wc -l

In [ ]:
!python scripts/prepare_data.py --ffhq /content/raw/ffhq --coco /content/raw/coco/val2017 --out /content/data/prepared

In [ ]:
Path(f"{DRIVE_ROOT}/data").mkdir(parents=True, exist_ok=True)
!tar -cf {DRIVE_ROOT}/data/prepared.tar -C /content/data prepared
!ls -lh {DRIVE_ROOT}/data